# Galilean IMU preintegration: NEES in inertial and rotating frames

This notebook compares GTSAM's Manifold, Tangent, Lie-group, and Galilean IMU preintegration backends under identical measurements and noise samples. Every NEES experiment evaluates both `ComponentWise` and `Logmap` IMU factor-error modes from the same preintegrated samples. Sections 2--6 retain the inertial-frame stress test with simultaneous body-frame acceleration and rotation. Section 7 adds a realistic powered-ascent benchmark in a rotating Earth frame and evaluates every backend both with `omegaCoriolis` specified and with it omitted. Sections 8--9 then test long-horizon uncertainty from a nonzero bias, first with fixed bias in 9D and then with bias random walk in the full 15D Combined PIM covariance.

The word *better* has two testable meanings here: smaller deterministic endpoint error at a fixed sample period, and statistical consistency measured by Normalized Estimation Error Squared (NEES). This experiment does not claim universal dominance, lower runtime, or an advantage when the assumed held-input model is a poor description of the sensor signal. See the companion [`GalileanImuFactor` guide](GalileanImuFactor.ipynb) for the complete left-invariant derivation.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/navigation/doc/GalileanImuFactorNEES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not in Colab

In [2]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from scipy.stats import chi2

import gtsam

np.set_printoptions(precision=4, suppress=True)
pio.renderers.default = "notebook_connected"

## 1. What NEES measures

For the factor residual $e\in\mathbb R^9$ and its predicted covariance $P$,

$$
\operatorname{NEES}=e^\mathsf{T}P^{-1}e.
$$

A consistent model has expected NEES equal to the residual dimension, nine. For $N$ independent trials, the exact 95% acceptance interval for the average NEES is

$$
\left[\frac{\chi^2_{0.025,9N}}{N},\frac{\chi^2_{0.975,9N}}{N}\right].
$$

Every backend below receives the same continuous-time accelerometer and gyroscope noise densities, the same sampled noise realization in each trial, the same zero bias, and the same exact endpoint. Gravity and integration noise are set to zero to isolate the preintegration discretization and its sensor-noise covariance. We evaluate the same samples twice: once with the component-wise $(SO(3)\times\mathbb R^6)$ error and once with the $SE_2(3)$ Logmap. These explicit modes are independent of the compile-time NavState optimization chart.

In [3]:
DURATION = 1.0
ACCELERATION = np.array([2.0, -1.0, 0.5])
ANGULAR_VELOCITY = np.array([1.2, -0.8, 2.0])
ACCELEROMETER_SIGMAS = np.array([0.03, 0.04, 0.05])
GYROSCOPE_SIGMAS = np.array([0.01, 0.015, 0.02])
BIAS = gtsam.imuBias.ConstantBias()
STATE_I = gtsam.NavState()

BACKENDS = {
    "Manifold": gtsam.PreintegratedImuMeasurementsManifold,
    "Tangent": gtsam.PreintegratedImuMeasurementsTangent,
    "Lie group": gtsam.PreintegratedImuMeasurementsLieGroup,
    "Galilean": gtsam.PreintegratedImuMeasurementsG,
}


ERROR_MODES = ("ComponentWise", "Logmap")
GTSAM_ERROR_MODES = {
    "ComponentWise": gtsam.ImuFactorErrorMode.ComponentWise,
    "Logmap": gtsam.ImuFactorErrorMode.Logmap,
}
MODE_PROBE = gtsam.PreintegrationParams(np.zeros(3))
assert (
    MODE_PROBE.getImuFactorErrorMode()
    == gtsam.ImuFactorErrorMode.Logmap
)
for mode in ERROR_MODES:
    MODE_PROBE.setImuFactorErrorMode(GTSAM_ERROR_MODES[mode])
    assert MODE_PROBE.getImuFactorErrorMode() == GTSAM_ERROR_MODES[mode]


def state_error(mode, mean_state, sampled_state):
    if mode == "Logmap":
        relative = mean_state.inverse().compose(sampled_state)
        return np.asarray(gtsam.NavState.Logmap(relative))
    if mode == "ComponentWise":
        rotation = gtsam.Rot3.Logmap(
            mean_state.attitude().between(sampled_state.attitude())
        )
        position = mean_state.attitude().unrotate(
            sampled_state.position() - mean_state.position()
        )
        velocity = mean_state.attitude().unrotate(
            sampled_state.velocity() - mean_state.velocity()
        )
        return np.concatenate((rotation, position, velocity))
    raise ValueError(f"Unknown IMU factor error mode: {mode}")


def physical_error(mean_state, sampled_state):
    rotation = gtsam.Rot3.Logmap(
        mean_state.attitude().between(sampled_state.attitude())
    )
    return np.concatenate(
        (
            rotation,
            sampled_state.position() - mean_state.position(),
            sampled_state.velocity() - mean_state.velocity(),
        )
    )



def make_params():
    params = gtsam.PreintegrationParams(np.zeros(3))
    params.setAccelerometerCovariance(
        np.diag(ACCELEROMETER_SIGMAS**2)
    )
    params.setGyroscopeCovariance(np.diag(GYROSCOPE_SIGMAS**2))
    params.setIntegrationCovariance(np.zeros((3, 3)))
    return params


PARAMS = make_params()

## 2. Independent closed-form endpoint

With constant body acceleration $a$, angular rate $\omega$, and duration $T$, the continuous held-input solution is the Galilean exponential

$$
\Upsilon(T)=\operatorname{Exp}_{\mathrm{Gal}(3)}(T(\omega,a,0,1)).
$$

This is the analytical solution of the continuous dynamics, not a fine-step numerical reference. We remove deterministic time and map $(R,v,p,T)$ into the `NavState` ordering $(R,p,v)`.

In [4]:
def exact_held_input_state(acceleration, angular_velocity, duration):
    tangent = np.zeros(10)
    tangent[:3] = angular_velocity * duration
    tangent[3:6] = acceleration * duration
    tangent[9] = duration
    delta = gtsam.Gal3.Expmap(tangent)
    return gtsam.NavState(
        delta.rotation(), delta.translation(), delta.velocity()
    )


TRUTH = exact_held_input_state(
    ACCELERATION, ANGULAR_VELOCITY, DURATION
)


def integrate(
    backend_type, accelerations, angular_velocities, dt,
    params=PARAMS, bias=BIAS,
):
    pim = backend_type(params, bias)
    for acceleration, angular_velocity in zip(
        accelerations, angular_velocities
    ):
        pim.integrateMeasurement(acceleration, angular_velocity, dt)
    return pim

## 3. Deterministic discretization error

First remove sensor noise and vary only the sample period. The three established backends use the same piecewise update for this trajectory and therefore overlap. Their error decreases linearly as the timestep shrinks. Galilean composition remains at floating-point precision because each held-input interval uses the exact coupled exponential.

In [5]:
SAMPLE_PERIODS = np.array([0.1, 0.05, 0.025, 0.0125])
deterministic = {
    name: {"position": [], "velocity": []}
    for name in BACKENDS
}

for dt in SAMPLE_PERIODS:
    steps = round(DURATION / dt)
    accelerations = np.tile(ACCELERATION, (steps, 1))
    angular_velocities = np.tile(ANGULAR_VELOCITY, (steps, 1))
    for name, backend_type in BACKENDS.items():
        pim = integrate(
            backend_type, accelerations, angular_velocities, dt
        )
        error = physical_error(
            TRUTH, pim.predict(STATE_I, BIAS)
        )
        deterministic[name]["position"].append(
            np.linalg.norm(error[3:6])
        )
        deterministic[name]["velocity"].append(
            np.linalg.norm(error[6:9])
        )

assert max(deterministic["Galilean"]["position"]) < 2e-12
assert max(deterministic["Galilean"]["velocity"]) < 4e-12
for name in ("Manifold", "Tangent", "Lie group"):
    assert deterministic[name]["position"][1] > 0.03
    assert deterministic[name]["velocity"][1] > 0.07

In [6]:
fig = go.Figure()
line_styles = {
    "Manifold": ("#777777", "solid"),
    "Tangent": ("#999999", "dash"),
    "Lie group": ("#bbbbbb", "dot"),
    "Galilean": ("#14866d", "solid"),
}
for name in BACKENDS:
    color, dash = line_styles[name]
    fig.add_scatter(
        x=SAMPLE_PERIODS,
        y=deterministic[name]["velocity"],
        mode="lines+markers",
        name=name,
        line=dict(color=color, dash=dash),
    )
fig.update_xaxes(type="log", title="IMU sample period (s)")
fig.update_yaxes(type="log", title="Velocity error norm (m/s)")
fig.update_layout(
    title="Held-input discretization error",
    template="plotly_white",
    legend_title_text="Backend",
)
fig.show()

## 4. Paired Monte Carlo NEES experiment

We now use a 20 Hz IMU for one second and add anisotropic white sensor noise. Continuous-time noise density $\sigma$ becomes sampled rate noise $\sigma/\sqrt{\Delta t}$. Each trial generates one noise sequence and feeds that identical sequence to all four backends, making this a paired comparison. Each backend supplies its own propagated `residualCovariance()`.

In [7]:
DT = 0.05
STEPS = round(DURATION / DT)
TRIALS = 3_000
SEED = 2231

rng = np.random.default_rng(SEED)
errors = {
    mode: {name: np.empty((TRIALS, 9)) for name in BACKENDS}
    for mode in ERROR_MODES
}
nees = {
    mode: {name: np.empty(TRIALS) for name in BACKENDS}
    for mode in ERROR_MODES
}
physical_errors = {
    name: np.empty((TRIALS, 9)) for name in BACKENDS
}

for trial in range(TRIALS):
    accelerometer_noise = rng.normal(size=(STEPS, 3)) * (
        ACCELEROMETER_SIGMAS / np.sqrt(DT)
    )
    gyroscope_noise = rng.normal(size=(STEPS, 3)) * (
        GYROSCOPE_SIGMAS / np.sqrt(DT)
    )
    accelerations = ACCELERATION + accelerometer_noise
    angular_velocities = ANGULAR_VELOCITY + gyroscope_noise

    for name, backend_type in BACKENDS.items():
        pim = integrate(
            backend_type, accelerations, angular_velocities, DT
        )
        sampled_state = pim.predict(STATE_I, BIAS)
        covariance = np.asarray(pim.residualCovariance())
        physical_errors[name][trial] = physical_error(
            TRUTH, sampled_state
        )
        for mode in ERROR_MODES:
            error = state_error(mode, TRUTH, sampled_state)
            errors[mode][name][trial] = error
            nees[mode][name][trial] = (
                error @ np.linalg.solve(covariance, error)
            )

## 5. Results

The expected band below is the exact chi-square interval for the average of 3,000 independent 9D NEES samples. The error bars on each point are empirical 95% confidence intervals for that backend's sampled mean. Position and velocity RMS values are norms within their three-dimensional blocks, so they retain physical units.

In [8]:
DIMENSION = 9
expected_interval = chi2.ppf(
    [0.025, 0.975], TRIALS * DIMENSION
) / TRIALS

physical_summary = {}
for name, backend_errors in physical_errors.items():
    physical_summary[name] = {
        "position_rmse": np.sqrt(
            np.mean(np.sum(backend_errors[:, 3:6] ** 2, axis=1))
        ),
        "velocity_rmse": np.sqrt(
            np.mean(np.sum(backend_errors[:, 6:9] ** 2, axis=1))
        ),
    }

summary = {mode: {} for mode in ERROR_MODES}
for mode in ERROR_MODES:
    for name in BACKENDS:
        backend_errors = errors[mode][name]
        values = nees[mode][name]
        summary[mode][name] = {
            "mean_nees": values.mean(),
            "mean_half_width": (
                1.96 * values.std(ddof=1) / np.sqrt(TRIALS)
            ),
            "mean_error_norm": np.linalg.norm(
                backend_errors.mean(axis=0)
            ),
        }

physical_rows = [
    "| Backend | Position RMS (m) | Velocity RMS (m/s) |",
    "|---|---:|---:|",
]
for name, values in physical_summary.items():
    physical_rows.append(
        f"| {name} | {values['position_rmse']:.4f} | "
        f"{values['velocity_rmse']:.4f} |"
    )
display(Markdown("\n".join(physical_rows)))

nees_rows = [
    "| Error mode | Backend | Mean NEES | Mean error norm |",
    "|---|---|---:|---:|",
]
for mode in ERROR_MODES:
    for name, values in summary[mode].items():
        nees_rows.append(
            f"| {mode} | {name} | {values['mean_nees']:.3f} | "
            f"{values['mean_error_norm']:.4f} |"
        )
display(Markdown("\n".join(nees_rows)))
print(
    "Expected 95% interval for mean NEES: "
    f"[{expected_interval[0]:.3f}, {expected_interval[1]:.3f}]"
)

| Backend | Position RMS (m) | Velocity RMS (m/s) |
|---|---:|---:|
| Manifold | 0.0578 | 0.1049 |
| Tangent | 0.0578 | 0.1049 |
| Lie group | 0.0578 | 0.1049 |
| Galilean | 0.0427 | 0.0767 |

| Error mode | Backend | Mean NEES | Mean error norm |
|---|---|---:|---:|
| ComponentWise | Manifold | 14.506 | 0.0818 |
| ComponentWise | Tangent | 14.511 | 0.0818 |
| ComponentWise | Lie group | 14.506 | 0.0818 |
| ComponentWise | Galilean | 8.958 | 0.0011 |
| Logmap | Manifold | 14.502 | 0.0818 |
| Logmap | Tangent | 14.507 | 0.0818 |
| Logmap | Lie group | 14.502 | 0.0818 |
| Logmap | Galilean | 8.959 | 0.0011 |

Expected 95% interval for mean NEES: [8.849, 9.152]


In [9]:
for mode in ERROR_MODES:
    for name in BACKENDS:
        assert np.isfinite(summary[mode][name]["mean_nees"])

for name in ("Manifold", "Tangent", "Lie group"):
    assert (
        physical_summary["Galilean"]["position_rmse"]
        < physical_summary[name]["position_rmse"]
    )
    assert (
        physical_summary["Galilean"]["velocity_rmse"]
        < physical_summary[name]["velocity_rmse"]
    )

In [10]:
names = list(BACKENDS)
mode_styles = {
    "ComponentWise": ("#b44b4b", "x"),
    "Logmap": ("#3569a8", "circle"),
}

fig = go.Figure()
fig.add_hrect(
    y0=expected_interval[0],
    y1=expected_interval[1],
    fillcolor="#14866d",
    opacity=0.14,
    line_width=0,
    annotation_text="95% consistency band",
    annotation_position="top left",
)
for mode, (color, symbol) in mode_styles.items():
    fig.add_scatter(
        x=names,
        y=[summary[mode][name]["mean_nees"] for name in names],
        mode="markers",
        name=mode,
        marker=dict(size=11, color=color, symbol=symbol),
        error_y=dict(
            type="data",
            array=[
                summary[mode][name]["mean_half_width"]
                for name in names
            ],
            visible=True,
        ),
        hovertemplate=(
            f"{mode}<br>%{{x}}: mean NEES %{{y:.3f}}"
            "<extra></extra>"
        ),
    )
fig.add_hline(y=DIMENSION, line_dash="dash", line_color="#333333")
fig.update_layout(
    title="Average 9D NEES under identical high-dynamic IMU samples",
    xaxis_title="Preintegration backend",
    yaxis_title="Mean NEES",
    yaxis_range=[
        0,
        max(
            summary[mode][name]["mean_nees"]
            for mode in ERROR_MODES for name in names
        ) + 1.5,
    ],
    template="plotly_white",
    legend_title_text="Error mode",
)
fig.show()

## 6. Interpretation and limits

The two residual definitions are evaluated on exactly the same predicted states and covariances, so their difference is purely the nonlinear chart. The physical RMS table is deliberately independent of that choice. With the fixed seed, Galilean preintegration remains closest to the expected NEES because its held-input mean removes most of the finite-rate position and velocity error; the comparison between the two rows for each backend shows the additional effect of the component-wise versus $SE_2(3)$ Logmap residual.

The endpoint table also separates consistency from accuracy: Galilean preintegration reduces both position and velocity RMS error, while all four backends have essentially the same rotation RMS error. The timestep sweep identifies the cause. The three established variants converge as the sampling interval shrinks, whereas the Galilean group law and exponential integrate the coupled held input exactly at every tested interval.

This result is deliberately scoped. At very high IMU rates the standard discretization error becomes negligible; with time-varying input inside a sample, all zero-order-hold methods inherit model error; and this notebook does not compare runtime, sensor-pose corrections, or full graph optimization. The later sections add rotating-Earth, fixed-bias, and bias-random-walk tests while preserving this original experiment.

## 7. Powered ascent in a rotating Earth frame

We now model the first four seconds of a high-power sounding-rocket ascent. This scale is grounded in a [Georgia Tech Experimental Rocketry flight](https://ae.gatech.edu/news/2024/07/ramblin-rocket-club-swarms-desert-annual-rocket-launch) that reached roughly 8,000 ft in six to seven seconds and Mach 1.8. The benchmark starts from rest, uses a constant 12 g measured specific force along the body vertical, and applies a modest $-0.04$ rad/s pitch rate. Its exact endpoint is about 861 m above the pad, traveling at 431 m/s with 9.2 degrees of pitch: a plausible early powered-ascent segment rather than an artificially high initial speed.

The local navigation frame is east-north-up at Spaceport America latitude, so the physical Earth-rate vector contains north and up components. For every backend we run two otherwise identical predictions: `Specified` supplies that vector through `omegaCoriolis`, while `Not specified` leaves the parameter absent. Each paired Monte Carlo trial uses the same IMU noise sequence in all eight predictions, and each prediction is evaluated with both error modes. The preintegrated body increment and covariance recursion are unchanged by Earth rotation; only endpoint prediction and residual assembly use the rotating-frame model.

In [11]:
EARTH_ANGULAR_SPEED = 7.292115e-5
SPACEPORT_LATITUDE = np.deg2rad(32.99)
EARTH_RATE = EARTH_ANGULAR_SPEED * np.array([
    0.0, np.cos(SPACEPORT_LATITUDE), np.sin(SPACEPORT_LATITUDE)
])
STANDARD_GRAVITY = 9.80665
ROTATING_GRAVITY = np.array([0.0, 0.0, -STANDARD_GRAVITY])
ROTATING_ACCELERATION = np.array([
    0.0, 0.0, 12.0 * STANDARD_GRAVITY
])
PITCH_RATE = -0.04
ROTATING_ANGULAR_VELOCITY = (
    EARTH_RATE + np.array([0.0, PITCH_RATE, 0.0])
)
ROTATING_DURATION = 4.0
ROTATING_DT = 0.05
ROTATING_STEPS = round(ROTATING_DURATION / ROTATING_DT)
ROTATING_TRIALS = 3_000
ROTATING_SEED = 2232
ROTATING_STATE_I = gtsam.NavState()
ROCKET_ACCELEROMETER_SIGMAS = ACCELEROMETER_SIGMAS.copy()
ROCKET_GYROSCOPE_SIGMAS = np.array([5e-4, 7.5e-4, 1e-3])


def make_rotating_params(with_coriolis):
    params = gtsam.PreintegrationParams(ROTATING_GRAVITY)
    params.setAccelerometerCovariance(
        np.diag(ROCKET_ACCELEROMETER_SIGMAS**2)
    )
    params.setGyroscopeCovariance(
        np.diag(ROCKET_GYROSCOPE_SIGMAS**2)
    )
    params.setIntegrationCovariance(np.zeros((3, 3)))
    if with_coriolis:
        params.setOmegaCoriolis(EARTH_RATE)
    return params


CORIOLIS_CONDITIONS = {
    "Not specified": False,
    "Specified": True,
}
ROTATING_PARAMS = {
    condition: make_rotating_params(enabled)
    for condition, enabled in CORIOLIS_CONDITIONS.items()
}

### Independent exact endpoint

The truth calculation below is a direct transcription of the paper's closed-form rotating-frame equation, not a call to a PIM's `predict`. It first evaluates the held body increment $U=(\Delta R,\Delta p,\Delta v)$ with the Galilean exponential. For $\theta=-\omega_E T$, it then forms $A=\operatorname{Exp}(\theta)$, $G^v=J_L(\theta)$, and $G^p=J_L(\theta)-\Gamma_2(\theta)$. In GTSAM's `NavState(R,p,v)` order,

$$
p_j=G^pgT^2+A(p_i+\bar v_iT+R_i\Delta p),\qquad
\bar v_j=G^vgT+A(\bar v_i+R_i\Delta v),
$$

with $\bar v_i=v_i+[\omega_E]_\times p_i$, $v_j=\bar v_j-[\omega_E]_\times p_j$, and $R_j=AR_i\Delta R$. This explicit block order is the only change from the paper's displayed $(R,v,p)$ convention.

In [12]:
def skew(vector):
    x, y, z = vector
    return np.array([[0.0, -z, y], [z, 0.0, -x], [-y, x, 0.0]])


def exact_rotating_state(
    state_i, acceleration, angular_velocity, duration, gravity, omega
):
    tangent = np.zeros(10)
    tangent[:3] = angular_velocity * duration
    tangent[3:6] = acceleration * duration
    tangent[9] = duration
    delta = gtsam.Gal3.Expmap(tangent)

    theta = -omega * duration
    kernels = gtsam.so3.DexpFunctor(theta)
    A = gtsam.Rot3.Expmap(theta).matrix()
    gamma_velocity = kernels.leftJacobian()
    gamma_position = gamma_velocity - kernels.Gamma().left()

    omega_cross = skew(omega)
    R_i = state_i.attitude().matrix()
    p_i = state_i.position()
    v_bar_i = state_i.velocity() + omega_cross @ p_i
    p_j = gamma_position @ gravity * duration**2 + A @ (
        p_i + v_bar_i * duration + R_i @ delta.translation()
    )
    v_bar_j = gamma_velocity @ gravity * duration + A @ (
        v_bar_i + R_i @ delta.velocity()
    )
    v_j = v_bar_j - omega_cross @ p_j
    R_j = A @ R_i @ delta.rotation().matrix()
    return gtsam.NavState(gtsam.Rot3(R_j), p_j, v_j)


ROTATING_TRUTH = exact_rotating_state(
    ROTATING_STATE_I,
    ROTATING_ACCELERATION,
    ROTATING_ANGULAR_VELOCITY,
    ROTATING_DURATION,
    ROTATING_GRAVITY,
    EARTH_RATE,
)

rotating_deterministic = {name: {} for name in BACKENDS}
nominal_accelerations = np.tile(
    ROTATING_ACCELERATION, (ROTATING_STEPS, 1)
)
nominal_angular_velocities = np.tile(
    ROTATING_ANGULAR_VELOCITY, (ROTATING_STEPS, 1)
)
for name, backend_type in BACKENDS.items():
    for condition, params in ROTATING_PARAMS.items():
        pim = integrate(
            backend_type,
            nominal_accelerations,
            nominal_angular_velocities,
            ROTATING_DT,
            params,
        )
        error = physical_error(
            ROTATING_TRUTH,
            pim.predict(ROTATING_STATE_I, BIAS),
        )
        rotating_deterministic[name][condition] = error

final_speed = np.linalg.norm(ROTATING_TRUTH.velocity())
final_altitude = ROTATING_TRUTH.position()[2]
final_pitch = np.rad2deg(ROTATING_TRUTH.attitude().rpy()[1])
print(
    f"Exact endpoint: altitude {final_altitude:.1f} m, "
    f"speed {final_speed:.1f} m/s, pitch {final_pitch:.1f} deg"
)

galilean_specified = rotating_deterministic["Galilean"][
    "Specified"
]
assert np.linalg.norm(galilean_specified) < 1e-8
assert np.linalg.norm(
    rotating_deterministic["Galilean"]["Not specified"]
) > 0.2
for name in ("Manifold", "Tangent", "Lie group"):
    specified_norm = np.linalg.norm(
        rotating_deterministic[name]["Specified"]
    )
    omitted_norm = np.linalg.norm(
        rotating_deterministic[name]["Not specified"]
    )
    assert specified_norm > 1.0
    assert omitted_norm > specified_norm

Exact endpoint: altitude 861.0 m, speed 431.1 m/s, pitch -9.2 deg


### Paired rotating-frame NEES

The deterministic checks separate the two errors. With Earth rate specified, the Galilean backend recovers the closed-form powered-ascent endpoint to floating-point precision, while the three established backends retain finite-rate position and velocity error from simultaneous thrust and pitch. Omitting Earth rate adds rotating-frame error to every backend. We then use a plausible lower-noise rocket IMU model and repeat 3,000 paired trials.

In [13]:
rotating_rng = np.random.default_rng(ROTATING_SEED)
rotating_errors = {
    mode: {
        name: {
            condition: np.empty((ROTATING_TRIALS, 9))
            for condition in CORIOLIS_CONDITIONS
        }
        for name in BACKENDS
    }
    for mode in ERROR_MODES
}
rotating_nees = {
    mode: {
        name: {
            condition: np.empty(ROTATING_TRIALS)
            for condition in CORIOLIS_CONDITIONS
        }
        for name in BACKENDS
    }
    for mode in ERROR_MODES
}
rotating_physical_errors = {
    name: {
        condition: np.empty((ROTATING_TRIALS, 9))
        for condition in CORIOLIS_CONDITIONS
    }
    for name in BACKENDS
}

for trial in range(ROTATING_TRIALS):
    accelerometer_noise = rotating_rng.normal(
        size=(ROTATING_STEPS, 3)
    ) * (ROCKET_ACCELEROMETER_SIGMAS / np.sqrt(ROTATING_DT))
    gyroscope_noise = rotating_rng.normal(
        size=(ROTATING_STEPS, 3)
    ) * (ROCKET_GYROSCOPE_SIGMAS / np.sqrt(ROTATING_DT))
    accelerations = ROTATING_ACCELERATION + accelerometer_noise
    angular_velocities = (
        ROTATING_ANGULAR_VELOCITY + gyroscope_noise
    )

    for name, backend_type in BACKENDS.items():
        for condition, params in ROTATING_PARAMS.items():
            pim = integrate(
                backend_type,
                accelerations,
                angular_velocities,
                ROTATING_DT,
                params,
            )
            predicted = pim.predict(ROTATING_STATE_I, BIAS)
            covariance = np.asarray(pim.residualCovariance())
            rotating_physical_errors[name][condition][trial] = (
                physical_error(ROTATING_TRUTH, predicted)
            )
            for mode in ERROR_MODES:
                error = state_error(mode, ROTATING_TRUTH, predicted)
                rotating_errors[mode][name][condition][trial] = error
                rotating_nees[mode][name][condition][trial] = (
                    error @ np.linalg.solve(covariance, error)
                )

In [14]:
rotating_expected_interval = chi2.ppf(
    [0.025, 0.975], ROTATING_TRIALS * DIMENSION
) / ROTATING_TRIALS

rotating_physical_summary = {name: {} for name in BACKENDS}
physical_rows = [
    "| Backend | Coriolis | Position RMS (m) | Velocity RMS (m/s) |",
    "|---|---|---:|---:|",
]
for name in BACKENDS:
    for condition in CORIOLIS_CONDITIONS:
        backend_errors = rotating_physical_errors[name][condition]
        rotating_physical_summary[name][condition] = {
            "position_rmse": np.sqrt(
                np.mean(np.sum(backend_errors[:, 3:6] ** 2, axis=1))
            ),
            "velocity_rmse": np.sqrt(
                np.mean(np.sum(backend_errors[:, 6:9] ** 2, axis=1))
            ),
        }
        result = rotating_physical_summary[name][condition]
        physical_rows.append(
            f"| {name} | {condition} | "
            f"{result['position_rmse']:.4f} | "
            f"{result['velocity_rmse']:.4f} |"
        )

display(Markdown("\n".join(physical_rows)))

rotating_summary = {
    mode: {name: {} for name in BACKENDS} for mode in ERROR_MODES
}
nees_rows = [
    "| Error mode | Backend | Coriolis | Mean NEES |",
    "|---|---|---|---:|",
]
for mode in ERROR_MODES:
    for name in BACKENDS:
        for condition in CORIOLIS_CONDITIONS:
            values = rotating_nees[mode][name][condition]
            mean = values.mean()
            half_width = 1.96 * values.std(ddof=1) / np.sqrt(
                ROTATING_TRIALS
            )
            rotating_summary[mode][name][condition] = {
                "mean_nees": mean,
                "mean_half_width": half_width,
                "mean_error_norm": np.linalg.norm(
                    rotating_errors[mode][name][condition].mean(axis=0)
                ),
            }
            nees_rows.append(
                f"| {mode} | {name} | {condition} | {mean:.3f} |"
            )

display(Markdown("\n".join(nees_rows)))
print(
    "Expected 95% interval for mean NEES: "
    f"[{rotating_expected_interval[0]:.3f}, "
    f"{rotating_expected_interval[1]:.3f}]"
)

for mode in ERROR_MODES:
    for name in BACKENDS:
        for condition in CORIOLIS_CONDITIONS:
            assert np.isfinite(
                rotating_summary[mode][name][condition]["mean_nees"]
            )

| Backend | Coriolis | Position RMS (m) | Velocity RMS (m/s) |
|---|---|---:|---:|
| Manifold | Not specified | 1.4079 | 0.8057 |
| Manifold | Specified | 1.2364 | 0.6859 |
| Tangent | Not specified | 1.4079 | 0.8057 |
| Tangent | Specified | 1.2364 | 0.6859 |
| Lie group | Not specified | 1.4079 | 0.8057 |
| Lie group | Specified | 1.2364 | 0.6859 |
| Galilean | Not specified | 0.8491 | 0.5317 |
| Galilean | Specified | 0.8213 | 0.5070 |

| Error mode | Backend | Coriolis | Mean NEES |
|---|---|---|---:|
| ComponentWise | Manifold | Not specified | 16.716 |
| ComponentWise | Manifold | Specified | 14.011 |
| ComponentWise | Tangent | Not specified | 16.716 |
| ComponentWise | Tangent | Specified | 14.011 |
| ComponentWise | Lie group | Not specified | 16.716 |
| ComponentWise | Lie group | Specified | 14.011 |
| ComponentWise | Galilean | Not specified | 9.664 |
| ComponentWise | Galilean | Specified | 9.090 |
| Logmap | Manifold | Not specified | 16.715 |
| Logmap | Manifold | Specified | 14.012 |
| Logmap | Tangent | Not specified | 16.715 |
| Logmap | Tangent | Specified | 14.012 |
| Logmap | Lie group | Not specified | 16.715 |
| Logmap | Lie group | Specified | 14.012 |
| Logmap | Galilean | Not specified | 9.663 |
| Logmap | Galilean | Specified | 9.090 |

Expected 95% interval for mean NEES: [8.849, 9.152]


In [15]:
fig = go.Figure()
fig.add_hrect(
    y0=rotating_expected_interval[0],
    y1=rotating_expected_interval[1],
    fillcolor="#14866d",
    opacity=0.14,
    line_width=0,
    annotation_text="95% consistency band",
    annotation_position="top left",
)
condition_symbols = {"Not specified": "x", "Specified": "circle"}
mode_colors = {"ComponentWise": "#b44b4b", "Logmap": "#3569a8"}
for mode in ERROR_MODES:
    for condition in CORIOLIS_CONDITIONS:
        fig.add_scatter(
            x=list(BACKENDS),
            y=[
                rotating_summary[mode][name][condition]["mean_nees"]
                for name in BACKENDS
            ],
            mode="markers",
            name=f"{mode} — {condition}",
            marker=dict(
                size=11, color=mode_colors[mode],
                symbol=condition_symbols[condition],
            ),
            error_y=dict(
                type="data",
                array=[
                    rotating_summary[mode][name][condition][
                        "mean_half_width"
                    ]
                    for name in BACKENDS
                ],
                visible=True,
            ),
            hovertemplate=(
                f"{mode} — {condition}<br>"
                "%{x}: mean NEES %{y:.3f}<extra></extra>"
            ),
        )
fig.add_hline(y=DIMENSION, line_dash="dash", line_color="#333333")
fig.update_layout(
    title="Powered-ascent NEES with and without Coriolis",
    xaxis_title="Preintegration backend",
    yaxis_title="Mean NEES",
    template="plotly_white",
    legend_title_text="Error mode — omegaCoriolis",
)
fig.show()

### Powered-ascent interpretation

The physical RMS table shows the Coriolis and integration effects without depending on a residual chart. The paired NEES rows then show how ComponentWise versus Logmap changes the statistical score for each of those same predictions. The three established methods retain finite-rate mean error while the rocket pitches under thrust, whereas Galilean preintegration removes that held-input error. Omitting `omegaCoriolis` worsens every backend by adding deterministic rotating-frame error that the sensor-noise covariance does not model.

This powered-ascent experiment exercises both effects together. Galilean composition removes held-input discretization error from simultaneous thrust and pitch, while `omegaCoriolis` supplies the exact rotating-frame lift, gravity kernels, and projection. The result is not obtained by amplifying Earth's rate or seeding an extreme initial velocity: the vehicle starts on the pad and reaches a flight state consistent with the early portion of a Georgia Tech high-power sounding-rocket launch.

## 8. Long-horizon uncertainty with nonzero bias

The preceding sections deliberately expose integration and rotating-frame errors. We now isolate the uncertainty representation emphasized by Brossard et al. The IMU follows ten seconds of smooth three-axis motion at 10 Hz with a fixed, nonzero accelerometer and gyroscope bias. Each backend receives the same anisotropic sensor-noise realization and is compared with its own noise-free discrete mean. Centering each distribution on its own mean removes held-input discretization error from this experiment, just as the paper recomputes its reference with the assumed discrete model.

For every backend we evaluate both the component-wise chart and $\log(\widehat T^{-1}T^n)$ in $SE_2(3)$. Thus this experiment asks both whether each propagated covariance describes its own noisy preintegrated measurements and how much of the result is caused by the residual definition rather than the preintegration backend.

In [16]:
UNCERTAINTY_DT = 0.1
UNCERTAINTY_STEPS = 100
UNCERTAINTY_TRIALS = 2_000
UNCERTAINTY_SEED = 2904
UNCERTAINTY_TIMES = (
    np.arange(UNCERTAINTY_STEPS) + 0.5
) * UNCERTAINTY_DT
UNCERTAINTY_ACCELERATIONS = np.column_stack(
    (
        0.8 + 0.35 * np.sin(0.37 * UNCERTAINTY_TIMES),
        -0.45 + 0.25 * np.cos(0.53 * UNCERTAINTY_TIMES),
        0.3 * np.sin(0.29 * UNCERTAINTY_TIMES),
    )
)
UNCERTAINTY_OMEGAS = np.column_stack(
    (
        0.35 * np.sin(0.41 * UNCERTAINTY_TIMES),
        -0.30 * np.cos(0.31 * UNCERTAINTY_TIMES),
        0.20 + 0.15 * np.sin(0.23 * UNCERTAINTY_TIMES),
    )
)
UNCERTAINTY_ACCELEROMETER_SIGMAS = np.array([0.25, 0.30, 0.35])
UNCERTAINTY_GYROSCOPE_SIGMAS = np.array([0.07, 0.09, 0.11])
NONZERO_BIAS = gtsam.imuBias.ConstantBias(
    np.array([0.08, -0.05, 0.03]),
    np.array([0.004, -0.006, 0.005]),
)


def make_uncertainty_params():
    params = gtsam.PreintegrationParams(np.zeros(3))
    params.setAccelerometerCovariance(
        np.diag(UNCERTAINTY_ACCELEROMETER_SIGMAS**2)
    )
    params.setGyroscopeCovariance(
        np.diag(UNCERTAINTY_GYROSCOPE_SIGMAS**2)
    )
    params.setIntegrationCovariance(np.zeros((3, 3)))
    return params


UNCERTAINTY_PARAMS = make_uncertainty_params()


def uncertainty_summary(nees, errors, dimension, trials):
    interval = chi2.ppf(
        [0.025, 0.975], trials * dimension
    ) / trials
    summary = {mode: {} for mode in ERROR_MODES}
    for mode in ERROR_MODES:
        for name in BACKENDS:
            values = nees[mode][name]
            summary[mode][name] = {
                "mean_nees": values.mean(),
                "median_nees": np.median(values),
                "mean_error_norm": np.linalg.norm(
                    errors[mode][name].mean(axis=0)
                ),
                "mean_half_width": (
                    1.96 * values.std(ddof=1) / np.sqrt(trials)
                ),
            }
    return summary, interval


def display_uncertainty_summary(summary, interval):
    rows = [
        "| Error mode | Backend | Mean NEES | Median NEES | Mean error norm |",
        "|---|---|---:|---:|---:|",
    ]
    for mode in ERROR_MODES:
        for name, values in summary[mode].items():
            rows.append(
                f"| {mode} | {name} | {values['mean_nees']:.3f} | "
                f"{values['median_nees']:.3f} | "
                f"{values['mean_error_norm']:.4f} |"
            )
    display(Markdown("\n".join(rows)))
    print(
        "Expected 95% interval for mean NEES: "
        f"[{interval[0]:.3f}, {interval[1]:.3f}]"
    )


def uncertainty_plot(summary, interval, dimension, title):
    figure = go.Figure()
    figure.add_hrect(
        y0=interval[0], y1=interval[1],
        fillcolor="#14866d", opacity=0.14, line_width=0,
        annotation_text="95% consistency band",
        annotation_position="top left",
    )
    names = list(BACKENDS)
    for mode, (color, symbol) in mode_styles.items():
        figure.add_scatter(
            x=names,
            y=[summary[mode][name]["mean_nees"] for name in names],
            mode="markers",
            name=mode,
            marker=dict(size=11, color=color, symbol=symbol),
            error_y=dict(
                type="data",
                array=[
                    summary[mode][name]["mean_half_width"]
                    for name in names
                ],
                visible=True,
            ),
            hovertemplate=(
                f"{mode}<br>%{{x}}: mean NEES %{{y:.3f}}"
                "<extra></extra>"
            ),
        )
    figure.add_hline(
        y=dimension, line_dash="dash", line_color="#333333"
    )
    figure.update_layout(
        title=title, xaxis_title="Preintegration backend",
        yaxis_title="Mean NEES", template="plotly_white",
        legend_title_text="Error mode",
    )
    figure.show()

In [17]:
nominal_accelerations = (
    UNCERTAINTY_ACCELERATIONS + NONZERO_BIAS.accelerometer()
)
nominal_omegas = UNCERTAINTY_OMEGAS + NONZERO_BIAS.gyroscope()
uncertainty_means = {}
for name, backend_type in BACKENDS.items():
    nominal_pim = integrate(
        backend_type, nominal_accelerations, nominal_omegas,
        UNCERTAINTY_DT, UNCERTAINTY_PARAMS, NONZERO_BIAS,
    )
    uncertainty_means[name] = nominal_pim.predict(
        STATE_I, NONZERO_BIAS
    )

uncertainty_rng = np.random.default_rng(UNCERTAINTY_SEED)
uncertainty_errors = {
    mode: {
        name: np.empty((UNCERTAINTY_TRIALS, 9)) for name in BACKENDS
    }
    for mode in ERROR_MODES
}
uncertainty_nees = {
    mode: {name: np.empty(UNCERTAINTY_TRIALS) for name in BACKENDS}
    for mode in ERROR_MODES
}

for trial in range(UNCERTAINTY_TRIALS):
    accelerations = nominal_accelerations + uncertainty_rng.normal(
        size=UNCERTAINTY_ACCELERATIONS.shape
    ) * (UNCERTAINTY_ACCELEROMETER_SIGMAS / np.sqrt(UNCERTAINTY_DT))
    omegas = nominal_omegas + uncertainty_rng.normal(
        size=UNCERTAINTY_OMEGAS.shape
    ) * (UNCERTAINTY_GYROSCOPE_SIGMAS / np.sqrt(UNCERTAINTY_DT))

    for name, backend_type in BACKENDS.items():
        pim = integrate(
            backend_type, accelerations, omegas, UNCERTAINTY_DT,
            UNCERTAINTY_PARAMS, NONZERO_BIAS,
        )
        predicted = pim.predict(STATE_I, NONZERO_BIAS)
        covariance = np.asarray(pim.residualCovariance())
        for mode in ERROR_MODES:
            error = state_error(mode, uncertainty_means[name], predicted)
            uncertainty_errors[mode][name][trial] = error
            uncertainty_nees[mode][name][trial] = (
                error @ np.linalg.solve(covariance, error)
            )

In [18]:
uncertainty_summary_values, uncertainty_interval = uncertainty_summary(
    uncertainty_nees, uncertainty_errors, 9, UNCERTAINTY_TRIALS
)
display_uncertainty_summary(
    uncertainty_summary_values, uncertainty_interval
)

for mode in ERROR_MODES:
    for name in BACKENDS:
        assert np.isfinite(
            uncertainty_summary_values[mode][name]["mean_nees"]
        )

uncertainty_plot(
    uncertainty_summary_values, uncertainty_interval, 9,
    "Long-horizon sensor-noise consistency with nonzero bias",
)

| Error mode | Backend | Mean NEES | Median NEES | Mean error norm |
|---|---|---:|---:|---:|
| ComponentWise | Manifold | 9.342 | 8.319 | 1.5652 |
| ComponentWise | Tangent | 9.365 | 8.360 | 1.6462 |
| ComponentWise | Lie group | 9.342 | 8.319 | 1.5652 |
| ComponentWise | Galilean | 9.347 | 8.338 | 1.5692 |
| Logmap | Manifold | 8.995 | 8.266 | 0.2156 |
| Logmap | Tangent | 8.996 | 8.286 | 0.3133 |
| Logmap | Lie group | 8.995 | 8.266 | 0.2156 |
| Logmap | Galilean | 8.993 | 8.253 | 0.2193 |

Expected 95% interval for mean NEES: [8.815, 9.187]


### Uncertainty interpretation

The paired rows show that the long-horizon consistency benefit follows the $SE_2(3)$ Logmap error definition across all four backends; it is not unique to the Lie-group preintegrator. Component-wise residuals can become overconfident and develop a larger nonzero sample mean over this long rotating interval. This separates Brossard's uncertainty-representation result from Galilean preintegration's held-input integration accuracy in the earlier sections.

## 9. Full state-bias uncertainty with bias random walk

Finally we use each backend's Combined PIM to propagate the full $15\times15$ covariance over $(R,p,v,b_a,b_\omega)$. Every trial begins at the same nonzero bias. After each measurement, accelerometer and gyroscope biases take an independent continuous-time random-walk step, and the next measurement contains the updated bias. The four backends receive the same sensor noise and the same bias path.

The residual appends $b_i-b_j$ to the nine navigation rows, matching `CombinedImuFactor`. This tests not just bias marginals but the propagated state-bias cross-covariances.

In [19]:
COMBINED_BACKENDS = {
    "Manifold": gtsam.PreintegratedCombinedMeasurementsManifold,
    "Tangent": gtsam.PreintegratedCombinedMeasurements,
    "Lie group": gtsam.PreintegratedCombinedMeasurementsLieGroup,
    "Galilean": gtsam.PreintegratedCombinedMeasurementsG,
}
BIAS_ACCELEROMETER_RW_SIGMAS = np.array([0.015, 0.020, 0.025])
BIAS_GYROSCOPE_RW_SIGMAS = np.array([0.002, 0.0025, 0.003])
BIAS_RW_SEED = 3904

combined_params = gtsam.PreintegrationCombinedParams(np.zeros(3))
combined_params.setAccelerometerCovariance(
    np.diag(UNCERTAINTY_ACCELEROMETER_SIGMAS**2)
)
combined_params.setGyroscopeCovariance(
    np.diag(UNCERTAINTY_GYROSCOPE_SIGMAS**2)
)
combined_params.setIntegrationCovariance(np.zeros((3, 3)))
combined_params.setBiasAccCovariance(
    np.diag(BIAS_ACCELEROMETER_RW_SIGMAS**2)
)
combined_params.setBiasOmegaCovariance(
    np.diag(BIAS_GYROSCOPE_RW_SIGMAS**2)
)

combined_means = {}
for name, backend_type in COMBINED_BACKENDS.items():
    pim = backend_type(combined_params, NONZERO_BIAS)
    for acceleration, omega in zip(
        nominal_accelerations, nominal_omegas
    ):
        pim.integrateMeasurement(
            acceleration, omega, UNCERTAINTY_DT
        )
    combined_means[name] = pim.predict(STATE_I, NONZERO_BIAS)

bias_rw_rng = np.random.default_rng(BIAS_RW_SEED)
combined_errors = {
    mode: {
        name: np.empty((UNCERTAINTY_TRIALS, 15))
        for name in BACKENDS
    }
    for mode in ERROR_MODES
}
combined_nees = {
    mode: {name: np.empty(UNCERTAINTY_TRIALS) for name in BACKENDS}
    for mode in ERROR_MODES
}

for trial in range(UNCERTAINTY_TRIALS):
    pims = {
        name: backend_type(combined_params, NONZERO_BIAS)
        for name, backend_type in COMBINED_BACKENDS.items()
    }
    accelerometer_bias = NONZERO_BIAS.accelerometer().copy()
    gyroscope_bias = NONZERO_BIAS.gyroscope().copy()

    for acceleration, omega in zip(
        UNCERTAINTY_ACCELERATIONS, UNCERTAINTY_OMEGAS
    ):
        measured_acceleration = (
            acceleration + accelerometer_bias
            + bias_rw_rng.normal(size=3)
            * UNCERTAINTY_ACCELEROMETER_SIGMAS
            / np.sqrt(UNCERTAINTY_DT)
        )
        measured_omega = (
            omega + gyroscope_bias
            + bias_rw_rng.normal(size=3)
            * UNCERTAINTY_GYROSCOPE_SIGMAS
            / np.sqrt(UNCERTAINTY_DT)
        )
        for pim in pims.values():
            pim.integrateMeasurement(
                measured_acceleration, measured_omega,
                UNCERTAINTY_DT,
            )
        accelerometer_bias += (
            bias_rw_rng.normal(size=3)
            * BIAS_ACCELEROMETER_RW_SIGMAS
            * np.sqrt(UNCERTAINTY_DT)
        )
        gyroscope_bias += (
            bias_rw_rng.normal(size=3)
            * BIAS_GYROSCOPE_RW_SIGMAS
            * np.sqrt(UNCERTAINTY_DT)
        )

    final_bias = gtsam.imuBias.ConstantBias(
        accelerometer_bias, gyroscope_bias
    )
    bias_error = np.asarray(
        NONZERO_BIAS.vector() - final_bias.vector()
    )
    for name, pim in pims.items():
        predicted = pim.predict(STATE_I, NONZERO_BIAS)
        covariance = np.asarray(pim.residualCovariance())
        for mode in ERROR_MODES:
            navigation_error = state_error(
                mode, combined_means[name], predicted
            )
            error = np.concatenate((navigation_error, bias_error))
            combined_errors[mode][name][trial] = error
            combined_nees[mode][name][trial] = (
                error @ np.linalg.solve(covariance, error)
            )

In [20]:
combined_summary, combined_interval = uncertainty_summary(
    combined_nees, combined_errors, 15, UNCERTAINTY_TRIALS
)
display_uncertainty_summary(combined_summary, combined_interval)

for mode in ERROR_MODES:
    for name in BACKENDS:
        assert np.isfinite(combined_summary[mode][name]["mean_nees"])

uncertainty_plot(
    combined_summary, combined_interval, 15,
    "Combined state-bias consistency with bias random walk",
)

| Error mode | Backend | Mean NEES | Median NEES | Mean error norm |
|---|---|---:|---:|---:|
| ComponentWise | Manifold | 15.345 | 14.448 | 1.5274 |
| ComponentWise | Tangent | 15.380 | 14.567 | 1.5975 |
| ComponentWise | Lie group | 15.345 | 14.448 | 1.5274 |
| ComponentWise | Galilean | 15.352 | 14.444 | 1.5392 |
| Logmap | Manifold | 15.011 | 14.310 | 0.1369 |
| Logmap | Tangent | 15.027 | 14.329 | 0.3117 |
| Logmap | Lie group | 15.011 | 14.310 | 0.1369 |
| Logmap | Galilean | 15.010 | 14.333 | 0.1390 |

Expected 95% interval for mean NEES: [14.761, 15.241]


### Bias-random-walk interpretation

The paired 15D results include the same sampled bias paths and the same state-bias covariance for both modes. The comparison therefore isolates the navigation residual chart while retaining all bias marginals and cross-covariances. Any consistency improvement shared by the four Logmap rows is the uncertainty-representation advantage discussed by Brossard; it is distinct from Galilean preintegration's exact held-input mean and from rotating-Earth compensation.

## 10. Logmap error, NEES, and backend recommendation

Fresh IMU parameters now select the $SE_2(3)$ Logmap error by default. The tables below compare all four backends under that regime. Bold values mark minimum physical or mean-residual errors and NEES values closest to the theoretical mean; powered-ascent values are compared within each Earth-rate condition.

In [21]:
def bold_min(value, values, digits):
    text = f"{value:.{digits}f}"
    return f"**{text}**" if np.isclose(value, min(values)) else text


def bold_nees(value, values, target):
    text = f"{value:.3f}"
    best = min(abs(candidate - target) for candidate in values)
    return f"**{text}**" if np.isclose(abs(value - target), best) else text


names = list(BACKENDS)
position_values = [physical_summary[name]["position_rmse"] for name in names]
velocity_values = [physical_summary[name]["velocity_rmse"] for name in names]
residual_values = [summary["Logmap"][name]["mean_error_norm"] for name in names]
nees_values = [summary["Logmap"][name]["mean_nees"] for name in names]
rows = [
    "### One-second inertial stress test",
    "",
    "| Backend | Position RMS (m) | Velocity RMS (m/s) | Mean Logmap residual norm | Mean NEES |",
    "|---|---:|---:|---:|---:|",
]
for name in names:
    physical = physical_summary[name]
    logmap = summary["Logmap"][name]
    rows.append(
        f"| {name} | {bold_min(physical['position_rmse'], position_values, 4)} "
        f"| {bold_min(physical['velocity_rmse'], velocity_values, 4)} "
        f"| {bold_min(logmap['mean_error_norm'], residual_values, 4)} "
        f"| {bold_nees(logmap['mean_nees'], nees_values, 9)} |"
    )

rows.extend(["", "### Powered ascent in a rotating Earth frame", "",
    "| Backend | Earth rate | Position RMS (m) | Velocity RMS (m/s) | Mean NEES |",
    "|---|---|---:|---:|---:|"])
for name in names:
    for condition in CORIOLIS_CONDITIONS:
        position_candidates = [
            rotating_physical_summary[candidate][condition]["position_rmse"]
            for candidate in names
        ]
        velocity_candidates = [
            rotating_physical_summary[candidate][condition]["velocity_rmse"]
            for candidate in names
        ]
        nees_candidates = [
            rotating_summary["Logmap"][candidate][condition]["mean_nees"]
            for candidate in names
        ]
        physical = rotating_physical_summary[name][condition]
        mean_nees = rotating_summary["Logmap"][name][condition]["mean_nees"]
        rows.append(
            f"| {name} | {condition} "
            f"| {bold_min(physical['position_rmse'], position_candidates, 4)} "
            f"| {bold_min(physical['velocity_rmse'], velocity_candidates, 4)} "
            f"| {bold_nees(mean_nees, nees_candidates, 9)} |"
        )

uncertainty_residuals = [
    uncertainty_summary_values["Logmap"][name]["mean_error_norm"]
    for name in names
]
uncertainty_nees_values = [
    uncertainty_summary_values["Logmap"][name]["mean_nees"] for name in names
]
combined_residuals = [
    combined_summary["Logmap"][name]["mean_error_norm"] for name in names
]
combined_nees_values = [
    combined_summary["Logmap"][name]["mean_nees"] for name in names
]
rows.extend(["", "### Long-horizon uncertainty", "",
    "The residual norms are norms of sample means in mixed tangent units, not physical RMSE values.", "",
    "| Backend | 9D mean residual norm | 9D NEES | 15D mean residual norm | 15D NEES |",
    "|---|---:|---:|---:|---:|"])
for name in names:
    uncertainty = uncertainty_summary_values["Logmap"][name]
    combined = combined_summary["Logmap"][name]
    rows.append(
        f"| {name} "
        f"| {bold_min(uncertainty['mean_error_norm'], uncertainty_residuals, 4)} "
        f"| {bold_nees(uncertainty['mean_nees'], uncertainty_nees_values, 9)} "
        f"| {bold_min(combined['mean_error_norm'], combined_residuals, 4)} "
        f"| {bold_nees(combined['mean_nees'], combined_nees_values, 15)} |"
    )

display(Markdown("\n".join(rows)))

### One-second inertial stress test

| Backend | Position RMS (m) | Velocity RMS (m/s) | Mean Logmap residual norm | Mean NEES |
|---|---:|---:|---:|---:|
| Manifold | 0.0578 | 0.1049 | 0.0818 | 14.502 |
| Tangent | 0.0578 | 0.1049 | 0.0818 | 14.507 |
| Lie group | 0.0578 | 0.1049 | 0.0818 | 14.502 |
| Galilean | **0.0427** | **0.0767** | **0.0011** | **8.959** |

### Powered ascent in a rotating Earth frame

| Backend | Earth rate | Position RMS (m) | Velocity RMS (m/s) | Mean NEES |
|---|---|---:|---:|---:|
| Manifold | Not specified | 1.4079 | 0.8057 | 16.715 |
| Manifold | Specified | 1.2364 | 0.6859 | 14.012 |
| Tangent | Not specified | 1.4079 | 0.8057 | 16.715 |
| Tangent | Specified | 1.2364 | 0.6859 | 14.012 |
| Lie group | Not specified | 1.4079 | 0.8057 | 16.715 |
| Lie group | Specified | 1.2364 | 0.6859 | 14.012 |
| Galilean | Not specified | **0.8491** | **0.5317** | **9.663** |
| Galilean | Specified | **0.8213** | **0.5070** | **9.090** |

### Long-horizon uncertainty

The residual norms are norms of sample means in mixed tangent units, not physical RMSE values.

| Backend | 9D mean residual norm | 9D NEES | 15D mean residual norm | 15D NEES |
|---|---:|---:|---:|---:|
| Manifold | **0.2156** | 8.995 | **0.1369** | 15.011 |
| Tangent | 0.3133 | **8.996** | 0.3117 | 15.027 |
| Lie group | **0.2156** | 8.995 | **0.1369** | 15.011 |
| Galilean | 0.2193 | 8.993 | 0.1390 | **15.010** |

### Recommendation

Use **Logmap** for the IMU factor-error mode unless reproducing historical ComponentWise behavior is an explicit requirement. `Legacy` remains available as the historical backend-dependent opt-in.

| Preintegration backend | Recommended use | Evidence and caution |
|---|---|---|
| **Galilean** | High dynamics, simultaneous rotation and acceleration, lower IMU rates, longer preintegration intervals, or rotating-Earth navigation with `omegaCoriolis` configured. | It has the lowest physical errors and consistent Logmap NEES in every correctly modeled experiment here. It is newer than the established backends, so confirm required wrapper, serialization, and downstream feature coverage. |
| **Tangent** | General-purpose production use, existing Tangent-based systems, or workflows requiring PIM `mergeWith`. | Its long-horizon Logmap NEES is consistent, but it retains the finite-rate coupled-motion error visible at 20 Hz. |
| **Manifold** | Reproducing the original on-manifold formulation or maintaining historically validated results. | Its Logmap NEES is consistent, but this benchmark shows no accuracy advantage over Tangent or Lie-group preintegration. |
| **Lie group** | Work that specifically requires applying each increment with the $SE_2(3)$ group exponential or aligning with a Lie-group derivation. | Its results match Manifold in these experiments; select it for formulation semantics rather than an assumed NEES advantage. |

Factor topology is a separate decision. Use `CombinedImuFactor` when bias random walk and state-bias cross-covariances belong in one 15D factor; use `ImuFactor` or `ImuFactor2` when bias evolution is modeled separately.